<a href="https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/exercicios/aula1-exercicio2-classes-com-mascara.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 1, exercício 2: Classes com máscara

Monte um conjunto (`set`) com os **nomes** das classes que ganharam máscara.

O exercício é da aula [1. Segmentação em imagem](https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/notebooks/01-segmentacao-em-imagem.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara os dados, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q ultralytics

In [ ]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mascaras_originais(resultado):
    """Máscaras no tamanho da foto, como array booleano (n, altura, largura).

    A rede responde numa grade menor que a imagem. Aqui a máscara é esticada
    de volta ao tamanho original, para a área bater com os pixels da foto.
    """
    import cv2
    import numpy as np

    altura, largura = resultado.orig_shape
    if resultado.masks is None or len(resultado.masks) == 0:
        return np.zeros((0, altura, largura), dtype=bool)
    saida = []
    for mascara in resultado.masks.data.cpu().numpy():
        mascara = cv2.resize(mascara, (largura, altura), interpolation=cv2.INTER_NEAREST)
        saida.append(mascara > 0.5)
    return np.stack(saida)


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

## Preparação

A preparação carrega o YOLOv8 de segmentação e roda na foto da enfermaria. Ficam prontos `resultado` e `mascaras`.

In [ ]:
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("yolov8n-seg.pt")
imagem = baixar("https://raw.githubusercontent.com/naubergois/curso-segmentacao-colab/main/midia/enfermaria.jpg", "enfermaria.jpg")
resultado = model(imagem, verbose=False)[0]
mascaras = mascaras_originais(resultado)

plt.figure(figsize=(6, 8))
plt.imshow(resultado.plot()[:, :, ::-1])
plt.axis("off")
plt.show()
print(len(mascaras), "máscaras")

In [ ]:
# @title Correção automática (rode, não precisa ler)
import numpy as np

_cls = resultado.boxes.cls.int().tolist()
_conf = resultado.boxes.conf.tolist()
_xyxy = resultado.boxes.xyxy.tolist()


def _indices(nome):
    return [i for i, c in enumerate(_cls) if model.names[c] == nome]


def _fracao_leitos():
    uniao = np.any(mascaras[_indices("bed")], axis=0)
    return float(uniao.mean())


def _n_leitos():
    indice = [i for i, nome in model.names.items() if nome == "bed"][0]
    r = model(imagem, classes=[indice], verbose=False)[0]
    return 0 if r.masks is None else len(r.masks)


VERIFICAR = {
    1: lambda r: int(r) == len(mascaras),
    2: lambda r: set(r) == {model.names[c] for c in _cls},
    3: lambda r: abs(float(r) - max(_conf)) < 1e-3,
    4: lambda r: int(r) == int(mascaras.sum(axis=(1, 2)).max()),
    5: lambda r: int(r) == _n_leitos(),
    6: lambda r: abs(float(r) - _fracao_leitos()) < 1e-3,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Monte um conjunto (`set`) com os **nomes** das classes que ganharam máscara.

*Dica:* model.names traduz o índice que está em resultado.boxes.cls.

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(2, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = {model.names[int(c)] for c in resultado.boxes.cls}

conferir(2, resposta)

## O que fica

Próximo: [Aula 1, exercício 3: A máscara mais segura](https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/exercicios/aula1-exercicio3-a-mascara-mais-segura.ipynb).